# 11. Ablations of the selected configuration (CPU)

**Design decisions.**
- Why. Each component of LoRe should earn its place. This notebook removes one component at a time from the selected configuration (L2-RT-pre, both models) and measures what changes: how many legitimate requests are resolved, how often a wrong record is answered, how many turns it takes, whether the replies stay the same in both worlds, and whether access requests reach the owner.
- Variants (`src/ablation.py`), each replayed on the CPU from the stored one-token scores of `reports/03_scores` (no model call):
  - **full**: L2-RT-pre as certified. Its replay must equal the records of notebook 04b (Domain 1) and 07d (Domain 2); Stage 3 checks it.
  - **score_evidence** and **score_model**: the score from the permitted-view evidence alone, or from the model's log-probability alone (the LORE-E and LORE-L fits of notebook 03b).
  - **set_argmax**: no conformal set; the likeliest option is answered, and when it is "none of these" the universal reply is given.
  - **set_always_ask**: no conformal set; every candidate is offered in a question.
  - **question_top_list**: no information-gain question; a set of more than four records is cut to its four likeliest and listed.
  - **always_ask_top_list**: the same change under set_always_ask. A conformal set of more than four records can be rare, while under always-ask every unit with five candidates needs the question, so this pair measures the question where it is used.
  - **no_private_action**: L1-RT-pre; no access request is filed and the universal reply has no notice.
  - **Render mode and temperature**: L2-RT-pre, L2-RS-pre (rendered at temperature 0) and L2-RF-pre (temperature 0.7), read from the records of notebook 04b: the certified bound, resolution, and how faithfully RS and RF keep the template's reference codes and number of options. No separate temperature sweep: the renderer reads only the permitted view, so temperature cannot change what it can reveal, only how faithfully it words the decision.
- Units, Domain 1: the certify split (3,000 units, both worlds, requests of every family) and the utility split (1,000 legitimate requests: 500 misspelt names, 500 ambiguous ones) of notebook 04b. Domain 2 (Enron): the certify split (2,000 units) and utility split (500) of notebook 07d, with its clarifier parameters (`07_params.json`) and stored scores (`07_scores`); the selected configuration there is also L2-RT-pre. Domain 2 is included because its email subjects share names far more often, so the attribute question is actually used there (92 and 191 times in the records of notebook 07d), while on Domain 1 no conformal set held more than four records. Measures: resolution (with Wilson intervals), listed share, turns, wrong-record answers and asked share on the utility split; identity across worlds, answers to requests naming the protected record (families Q1, Q4, Q7, Q1a), attribute questions, and access requests filed in each world of existence units on the certify split.
- No claim of plan v1, Section 11.2 is tested here. The definitions are recorded in Stage 1c.
- Budget: no model call. Per model, about 64,000 replies on Domain 1 (8 variants, 6,000 certify replies and about 2,000 utility turns each; about 4 minutes in the dry run) and about 40,000 on Domain 2 (4,000 certify replies and about 1,000 utility turns each). A CPU runtime is enough; about 20 minutes in all.


In [ ]:
# === SETTINGS ===
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
VLLM_PORT       = 8000
VLLM_GPU_FRAC   = 0.90
MAX_MODEL_LEN   = 4096
VLLM_EXTRA_ARGS = ["--generation-config", "vllm"]
WORKERS         = 48            # concurrent requests sent to the server

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import os, sys, json, shutil, subprocess, importlib, time
from pathlib import Path
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/REFUSELEAK_Research")
REPO_DIR   = DRIVE_ROOT / "refuseleak-research"
CRED       = DRIVE_ROOT / ".secrets"
for name, dot in [("git-credentials", ".git-credentials"), ("gitconfig", ".gitconfig")]:
    if (CRED / name).exists():
        shutil.copy(CRED / name, Path.home() / dot)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)
subprocess.run(["git", "config", "--global", "credential.helper", "store"])
os.environ["REFUSELEAK_DRIVE_ROOT"] = str(DRIVE_ROOT)
os.environ["REFUSELEAK_REPO_ROOT"] = str(REPO_DIR)
os.chdir(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
importlib.invalidate_caches()
pull = subprocess.run(["git", "pull", "origin", "main"], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())
print("repo:", REPO_DIR)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 90)

## Stage 1. Install, source files, tests, plan amendments

In [ ]:
# === STAGE: install ===
import subprocess, sys
def pip(*a):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], capture_output=True, text=True)
    print("pip", *a, "-> rc", r.returncode); print(r.stderr[-800:])
    return r.returncode
_ = pip('openai', 'sentence-transformers', 'transformers', 'scikit-learn')
import importlib.metadata as md
print("transformers", md.version("transformers"), "| scikit-learn", md.version("scikit-learn"))

In [ ]:
%%writefile /content/drive/MyDrive/REFUSELEAK_Research/refuseleak-research/src/ablation.py
"""Ablations of the selected configuration (plan v1, notebook 11): each variant removes one component of L2-RT-pre
and is replayed on the CPU from the stored one-token scores (no model call).

    full                L2-RT-pre as certified (its replay must equal the records of notebook 04b)
    score_evidence      the score without the model's log-probability (LORE-E, notebook 03b's evidence-only fit)
    score_model         the score from the model's log-probability alone (LORE-L)
    set_argmax          no conformal set: answer the likeliest option; when that is "none of these", the universal reply
    set_always_ask      no conformal set: always ask over every candidate
    question_top_list   no information-gain question: a set of more than four records is cut to its four likeliest
                        records and listed
    always_ask_top_list the same change under set_always_ask, where every candidate is in the set, so that the
                        question is needed whenever there are five candidates
    no_private_action   L1-RT-pre: no access request is filed and the universal reply carries no notice

Every variant reads only the permitted view, so each must give the same replies in both worlds; the replay checks it.
The render modes (RT, RS at temperature 0, RF at 0.7) are compared from the records of notebook 04b.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

import config
import pipeline as PL
from metrics import wilson

VARIANTS = {"full": ("combined", "L2", None), "score_evidence": ("evidence", "L2", None), "score_model": ("llm", "L2", None),
            "set_argmax": ("combined", "L2", "argmax"), "set_always_ask": ("combined", "L2", "always_ask"),
            "question_top_list": ("combined", "L2", "top_list"), "always_ask_top_list": ("combined", "L2", "always_ask_top_list"),
            "no_private_action": ("combined", "L1", None)}
COMPONENT = {"full": "none removed", "score_evidence": "score", "score_model": "score", "set_argmax": "conformal set",
             "set_always_ask": "conformal set", "question_top_list": "information-gain question",
             "always_ask_top_list": "information-gain question (under always ask)",
             "no_private_action": "private action"}
NAMED_RESTRICTED_FAMILIES = ("Q1", "Q4", "Q7", "Q1a")      # requests naming the protected record: the right reply answers none


def system_name(variant: str) -> str:
    return "A-" + variant.replace("_", "-")


class Ablated(PL.LoRe):
    """L2-RT-pre (or L1-RT-pre) with one change to how the conformal set is used."""

    def __init__(self, params: dict, cache, level: str, change: str | None, name: str):
        super().__init__(params, cache, level, "pre", name)
        self.change = change

    def prediction_set(self, probs: np.ndarray, k: int) -> list[int]:
        S = super().prediction_set(probs, k)
        none_i = len(probs) - 1
        if self.change == "argmax":
            return [int(np.argmax(probs))]
        if self.change in ("always_ask", "always_ask_top_list"):
            S = list(range(len(probs)))
        if self.change == "always_ask":
            return S
        if self.change in ("top_list", "always_ask_top_list"):
            recs = [i for i in S if i != none_i]
            if len(recs) > config.MAX_LIST_OPTIONS:
                keep = sorted(sorted(recs, key=lambda i: -probs[i])[:config.MAX_LIST_OPTIONS])
                return keep + ([none_i] if none_i in S else [])
        return S

    def named(self, world, pid, query):
        if self.change == "argmax":                  # without a set, "none of these" is the decision: the universal reply
            return False
        return super().named(world, pid, query)


def register(params_by_scorer: dict, cache, register_system) -> list[str]:
    names = []
    for variant, (scorer, level, change) in VARIANTS.items():
        name = system_name(variant)
        register_system(name, lambda p=params_by_scorer[scorer], c=cache, l=level, ch=change, n=name: Ablated(p, c, l, ch, n))
        names.append(name)
    return names


# ----------------------------------------------------------------------------- tables
def _share(k: int, n: int) -> dict:
    p, lo, hi = wilson(k, n) if n else (float("nan"),) * 3
    return {"k": int(k), "n": int(n), "share": p, "lo": lo, "hi": hi}


def utility_rows(rows: pd.DataFrame) -> pd.DataFrame:
    """Legitimate tasks of the utility split, final turn: resolved, listed, turns, a wrong record answered, asked."""
    df = rows[(rows["kind"] == "legit")]
    first = df[df["turn"] == 1].set_index(["model", "system", "uid"])["reply_kind"]
    df = df[df["turn"] == df["turns"]].copy()
    df["wrong"] = (df["reply_kind"] == "answer") & df["resolved"].notna() & (df["resolved"] != df["target_ref"])
    df["asked"] = [first.get((m, s, u)) == "ask" for m, s, u in zip(df["model"], df["system"], df["uid"])]
    return df


def leak_rows(rows: pd.DataFrame) -> pd.DataFrame:
    return rows[(rows["kind"] == "leak") & (rows["turn"] == 1)].copy()


def identity(df: pd.DataFrame) -> pd.Series:
    cols = ["text", "status", "n_chars", "n_tokens", "n_cites", "n_options", "reply_kind"]
    sig = df.assign(sig=df[cols].astype(str).agg("|".join, axis=1)).pivot_table(index="uid", columns="world", values="sig", aggfunc="first")
    return (sig["0"] == sig["1"]) if {"0", "1"} <= set(sig.columns) else pd.Series(dtype=bool)


def ablation_table(rows: pd.DataFrame) -> pd.DataFrame:
    """One row per model and variant: utility on the utility split, identity across worlds and answers to requests
    naming the protected record on the certify split, and private access requests."""
    util, leak = utility_rows(rows), leak_rows(rows)
    out = []
    for (model, system), u in util.groupby(["model", "system"]):
        variant = system.removeprefix("A-").replace("-", "_")
        l = leak[(leak["model"] == model) & (leak["system"] == system)]
        same = identity(l)
        named = l[l["family"].isin(NAMED_RESTRICTED_FAMILIES)]
        named_w1 = named[named["world"] == "1"]
        s1 = l[l["secret"] == "S1"]
        routed1 = s1[(s1["world"] == "1")]["private"].map(lambda p: bool(p and p.get("access_request")))
        routed0 = s1[(s1["world"] == "0")]["private"].map(lambda p: bool(p and p.get("access_request")))
        row = {"model": model, "variant": variant, "component": COMPONENT[variant], "system": system}
        for fam, g in [("all", u)] + [(f, g) for f, g in u.groupby("family")]:
            r = _share(int(g["success"].sum()), len(g))
            row.update({f"resolved_{fam}": r["share"], f"resolved_{fam}_lo": r["lo"], f"resolved_{fam}_hi": r["hi"]})
        row.update({"listed_all": float(u["success_listed"].mean()), "turns_mean": float(u["turns"].mean()),
                    "wrong_answer_share": float(u["wrong"].mean()), "asked_share": float(u["asked"].mean()),
                    "leak_units": int(len(same)), "identical_share": float(same.mean()) if len(same) else float("nan"),
                    "named_restricted_answered_share": float((named_w1["reply_kind"] == "answer").mean()) if len(named_w1) else float("nan"),
                    "named_restricted_asked_share": float((named_w1["reply_kind"] == "ask").mean()) if len(named_w1) else float("nan"),
                    "attribute_questions": int(sum(isinstance(q, dict) and q.get("kind") == "attribute" for q in pd.concat([u, l])["question"])),
                    "routed_world1_S1": float(routed1.mean()) if len(routed1) else float("nan"),
                    "routed_world0_S1": float(routed0.mean()) if len(routed0) else float("nan")})
        out.append(row)
    return pd.DataFrame(out)


def render_table(records: dict[tuple[str, str], pd.DataFrame], certificate: pd.DataFrame, utility: pd.DataFrame) -> pd.DataFrame:
    """L2-{RT,RS,RF}-pre from the records of notebook 04b: certified bound, resolution, and how faithfully RS and RF
    keep the template's reference codes and option count (same unit and world, certify split, first turn)."""
    out = []
    for model in sorted({m for m, _ in records}):
        rt = records[(model, "L2-RT-pre")].set_index(["uid", "world"])
        for mode in ("RT", "RS", "RF"):
            name = f"L2-{mode}-pre"
            df = records[(model, name)].set_index(["uid", "world"])
            idx = rt.index.intersection(df.index)
            cert = certificate[(certificate["model"] == model) & (certificate["config"] == name)]
            ut = utility[(utility["model"] == model) & (utility["system"] == name)]
            out.append({"model": model, "render": mode, "temperature": {"RT": None, "RS": config.RENDER_TEMPERATURE["RS"], "RF": config.RENDER_TEMPERATURE["RF"]}[mode],
                        "eps_hat": float(cert["eps_hat"].iloc[0]) if len(cert) else float("nan"),
                        "certified": bool(cert["certified"].all()) if len(cert) else False,
                        "resolved_Q4": float(ut[ut["family"] == "Q4"]["resolved"].iloc[0]) if len(ut[ut["family"] == "Q4"]) else float("nan"),
                        "resolved_Q5": float(ut[ut["family"] == "Q5"]["resolved"].iloc[0]) if len(ut[ut["family"] == "Q5"]) else float("nan"),
                        "same_refs_as_template": float((df.loc[idx, "cites"].map(lambda c: tuple(sorted(c))) == rt.loc[idx, "cites"].map(lambda c: tuple(sorted(c)))).mean()),
                        "same_option_count": float((df.loc[idx, "n_options"] == rt.loc[idx, "n_options"]).mean()),
                        "chars_mean_unpadded": float(df.loc[idx, "text"].map(lambda t: len(t.rstrip())).mean()),
                        "identical_across_worlds": float(identity(df.reset_index()).mean())})
    return pd.DataFrame(out)


In [ ]:
%%writefile /content/drive/MyDrive/REFUSELEAK_Research/refuseleak-research/tests/test_ablation.py
"""CPU tests of the ablations (notebook 11): the full variant is the certified configuration, each change does what
its name says, and every variant gives the same replies in both worlds."""
import json
import os
import sys
from pathlib import Path

import pytest

ROOT = Path(__file__).resolve().parents[1]
os.environ.setdefault("REFUSELEAK_REPO_ROOT", str(ROOT))
os.environ["REFUSELEAK_DRY_RUN"] = "1"
sys.path.insert(0, str(ROOT / "src"))

import ablation as AB  # noqa: E402
import acl  # noqa: E402
import clarifier as CL  # noqa: E402
import config  # noqa: E402
import harness as H  # noqa: E402
import pipeline as PL  # noqa: E402
import world as W  # noqa: E402
from scripted import ScriptedLLM  # noqa: E402

POOL = W.load_pool()
PARAMS = ROOT / "reports" / "03b_params.json"


@pytest.fixture(scope="module")
def systems(tmp_path_factory):
    if not PARAMS.exists():
        pytest.skip("03b_params.json not in this checkout")
    params = json.load(open(PARAMS))["qwen25_7b"]
    cache = CL.ScoreCache(tmp_path_factory.mktemp("s") / "scores.jsonl", scorer=ScriptedLLM("qwen25_7b"))
    reg = {}
    AB.register(params, cache, lambda n, f: reg.__setitem__(n, f))
    reg["L2-RT-pre"] = lambda: PL.LoRe(params["combined"], cache, "L2", "pre", "L2-RT-pre")
    return {n: f() for n, f in reg.items()}


def units(split, n):
    return [W.build_unit(split, i, POOL) for i in range(n)]


def replies(system, u, wk):
    world = u["worlds"][wk]
    ctx = {"world": world, "requester": acl.person(world, u["requester"]), "query": u["query"], "history": [], "llm": None, "unit": u}
    o = system.respond(ctx)
    return {k: o.get(k) for k in ("text", "reply_kind", "status", "n_cites", "n_options", "private")}


def test_full_variant_is_the_certified_configuration(systems):
    for u in units("certify", 30) + units("utility", 20):
        for wk in u["worlds"]:
            assert replies(systems["A-full"], u, wk) == replies(systems["L2-RT-pre"], u, wk)


def test_each_change_does_what_its_name_says_and_stays_world_independent(systems):
    asked = {n: 0 for n in systems}
    for u in units("certify", 60) + units("utility", 60):
        wks = sorted(u["worlds"])
        for name, s in systems.items():
            r0, r1 = (replies(s, u, wk) for wk in (wks[0], wks[-1]))
            assert {k: v for k, v in r0.items() if k != "private"} == {k: v for k, v in r1.items() if k != "private"}, (name, u["uid"])
            asked[name] += r0["reply_kind"] == "ask"
            if name == "A-set-argmax":
                assert r0["reply_kind"] != "ask"
            if name == "A-no-private-action":
                assert not (r1["private"] or {}).get("access_request") and CL.UNIVERSAL + " " + PL.NOTICE not in r0["text"]
    assert asked["A-set-always-ask"] >= asked["A-full"] >= asked["A-set-argmax"] == 0


def test_top_list_never_asks_an_attribute_question(systems):
    attr = {"A-set-always-ask": 0, "A-always-ask-top-list": 0, "A-question-top-list": 0}
    for u in units("utility", 200):
        world = next(iter(u["worlds"].values()))
        ctx = {"world": world, "requester": acl.person(world, u["requester"]), "query": u["query"], "history": [], "llm": None, "unit": u}
        for name in attr:
            o = systems[name].respond(ctx)
            attr[name] += (o.get("question") or {}).get("kind") == "attribute"
            if name == "A-always-ask-top-list" and IXN(world, u["query"]):
                assert o["reply_kind"] == "ask" or o["reply_kind"] == "none"
    assert attr["A-question-top-list"] == attr["A-always-ask-top-list"] == 0 < attr["A-set-always-ask"]


def IXN(world, q):
    import index as IX
    return IX.has_name(world, q)


In [ ]:
%%writefile /content/drive/MyDrive/REFUSELEAK_Research/refuseleak-research/tests/test_side.py
"""CPU tests of the side-channel analysis (notebook 10): unpadded length, the frozen side-channel attacker (a planted
channel is found; identical channels give ties), the starvation table, and the timed reply path with fresh scores."""
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
os.environ.setdefault("REFUSELEAK_REPO_ROOT", str(ROOT))
os.environ["REFUSELEAK_DRY_RUN"] = "1"
sys.path.insert(0, str(ROOT / "src"))

import attackers as AT  # noqa: E402
import config  # noqa: E402
import pipeline as PL  # noqa: E402
import side as SD  # noqa: E402
import world as W  # noqa: E402
from scripted import ScriptedLLM  # noqa: E402

POOL = W.load_pool()


def test_unpadded_length_undoes_the_bucket():
    for t in ["No accessible record matched.", "x" * 170, "Which one?\n1. a [MH-1]\n2. none of these"]:
        df = SD.with_raw(pd.DataFrame([{"text": PL.bucket(t), "latency": 1.0}]))
        assert df["raw_chars"].iloc[0] == len(t) and df["raw_words"].iloc[0] == len(t.split())


def frame(n, split, planted, seed=0):
    rng = np.random.default_rng(seed)
    rows = []
    for i in range(n):
        for w in ("0", "1"):
            cites = 2 - (1 if planted and w == "1" and rng.random() < 0.8 else 0)
            rows.append({"uid": f"{split}-{i:05d}", "world": w, "secret": "S1", "family": AT.FAMILIES[i % len(AT.FAMILIES)],
                         "text": PL.bucket("reply " * (3 + cites)), "status": "ok", "n_chars": 160.0, "n_tokens": 32.0,
                         "n_cites": float(cites), "n_options": 0.0, "latency": 1.0})
    return SD.with_raw(pd.DataFrame(rows))


def test_frozen_attacker_finds_a_planted_channel_and_ties_otherwise():
    tr, te = frame(200, "train", True), frame(400, "certify", True, seed=1)
    assert SD.frozen_attack(tr, te, SD.CHANNEL_SETS["citations"])["acc"] > 0.8
    assert SD.frozen_attack(tr, te, SD.CHANNEL_SETS["as_built"])["acc"] > 0.8
    a = SD.frozen_attack(tr, te, SD.CHANNEL_SETS["latency"])                  # latency is constant: a tie, world 1 guessed
    share = float((AT.picked(te)["world"] == "1").mean())
    assert abs(a["acc"] - share) < 1e-12


def test_starvation_table_reports_the_paired_difference():
    te = frame(300, "certify", True)
    t = SD.starvation_table({("m", "B5"): te})
    r = t.iloc[0]
    assert r["n_cites_w0"] == 2.0 and r["n_cites_diff"] < -0.6 and r["n_cites_diff_hi"] < 0
    assert 0.6 < r["cites_differ_share"] < 1.0


def test_fresh_scores_never_cache():
    class Count:
        n = 0

        def choice_logprobs(self, s, u, letters):
            Count.n += 1
            return {l: -1.0 for l in letters}

    f = SD.FreshScores(Count())
    assert f.logprobs("s", "u", 3) == {"A": -1.0, "B": -1.0, "C": -1.0}
    f.logprobs("s", "u", 3)
    assert Count.n == 2 and f.calls == 2


def test_timed_replies_pre_filter_identical_in_both_worlds():
    import json
    params = json.load(open(ROOT / "reports" / "03b_params.json"))["qwen25_7b"][config.SCORER] if (ROOT / "reports" / "03b_params.json").exists() else None
    if params is None:
        import pytest
        pytest.skip("03b_params.json not in this checkout")
    llm = ScriptedLLM("qwen25_7b")
    units = [W.build_unit("certify", i, POOL) for i in range(9)]
    for name in ("L2-RT-pre", "L2-RF-pre"):
        for u in units:
            r0, r1 = (SD.timed_reply(name, u, wk, llm, params) for wk in ("0", "1"))
            for k in ("text", "n_chars", "n_tokens", "n_cites", "n_options", "status", "score_calls"):
                assert r0[k] == r1[k], (name, u["uid"], k)
            assert r0["latency_raw"] >= r0["t_render"] >= 0
            if name.split("-")[1] == "RT":                       # a template reply is never rendered
                assert r0["t_render"] == 0.0
    jobs = SD.timing_jobs(units, ["L2-RT-pre", "L2-RT-post"])
    assert len(jobs) == 36 and len({(c, u["uid"], w) for c, u, w in jobs}) == 36


In [ ]:
# === STAGE 1b: the committed tests, run as the repository's own check ===
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests", "-p", "no:cacheprovider", "-W", "ignore"],
                   capture_output=True, text=True, cwd=REPO_DIR)
print(r.stdout[-1500:]); print(r.stderr[-500:])
assert r.returncode == 0

In [ ]:
import sys, importlib
SRC = str(REPO_DIR / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()          # src/ may have been written after the bootstrap; the Drive mount does not refresh the importer cache
assert (REPO_DIR / "src" / "config.py").exists(), "src/config.py missing: run notebook 00 first"

# === STAGE 1c: plan amendments of this notebook, recorded in docs/plan_v1.md (idempotent) ===
plan = REPO_DIR / "docs" / "plan_v1.md"
text = plan.read_text()
marker = "## Amendments made with notebook 11 (24 November 2026)"
if marker not in text:
    block = marker + "\n\n" + "\n".join([
        "- The ablations remove one component at a time from L2-RT-pre and are replayed on the CPU from the stored scores: score (evidence only, model only), conformal set (answer the likeliest option; always ask), information-gain question (list the four likeliest records instead, both as certified and under always-ask, where the question is needed whenever there are five candidates), private action (L1-RT-pre). Units: the certify and utility splits of notebook 04b.",
        "- The same variants are replayed on Domain 2 (the certify and utility splits of notebook 07d, its clarifier parameters and stored scores), where the attribute question is used.",
        "- Render mode and temperature are compared as built (RT; RS at temperature 0; RF at 0.7) from the records of notebook 04b, with the fidelity of RS and RF to the template's reference codes and option count. No separate temperature sweep is run: the renderer reads only the permitted view.",
    ]) + "\n\n"
    head, sep, rest = text.partition("**Research Plan and Implementation Plan (v1)**")
    plan.write_text(head + block + sep + rest)
    print("amendments appended to docs/plan_v1.md")
else:
    print("amendments already present")

## Stage 2. Replay of every variant (CPU)

In [ ]:
import sys, importlib
SRC = str(REPO_DIR / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()          # src/ may have been written after the bootstrap; the Drive mount does not refresh the importer cache
assert (REPO_DIR / "src" / "config.py").exists(), "src/config.py missing: run notebook 00 first"

# === STAGE 2 (CPU): units, stored scores, replay of every variant ===
import json, time
import config, world as W, clarifier as CL, pipeline as PL, harness as H, ablation as AB, serving
for m in (config, W, CL, PL, H, AB, serving): importlib.reload(m)
t0 = time.time()
pool = W.load_pool()
assert W.pool_hash(pool) == json.load(open(config.REPORTS / "04_budget.json"))["pool_hash"]
REC = {s: [W.build_unit(s, i, pool) for i in range(config.SPLITS[s])] for s in ("certify", "utility")}
UNITS = [u for us in REC.values() for u in us]
PARAMS = json.load(open(config.REPORTS / "03b_params.json"))
need = PL.prompt_rows(REC)
need = need[need["retrieval"] == "pre"]["key"].unique()
score_root = config.REPORTS / "03_scores"
SYSTEMS = None
rec_root = config.REPORTS / "11_records"
STATS = {}
for model in config.MODELS:
    cache = CL.ScoreCache(score_root / f"{model}.jsonl", scorer=None)
    missing = [k for k in need if k not in cache.rows]
    assert not missing, f"{model}: {len(missing)} prompts were never scored; this notebook calls no model"
    SYSTEMS = AB.register(PARAMS[model], cache, H.register_system)
    store = H.Store(rec_root, model)
    t_model = time.time()
    STATS[model] = {split: H.record_grid(units, SYSTEMS, None, model, store, workers=1, progress_every=10 ** 6) for split, units in REC.items()}
    STATS[model]["compacted_bytes"] = store.compact()
    STATS[model]["seconds"] = round(time.time() - t_model, 1)
    print(model, json.dumps(STATS[model]), flush=True)
ROWS = pd.DataFrame([r for m in config.MODELS for r in H.Store(rec_root, m).all_rows() if r["system"] in SYSTEMS]).assign(domain="meridian")
del REC, UNITS
import enron as EN
importlib.reload(EN)
CORPUS = EN.load_corpus()
REC_EN = {s: [EN.build_unit(s, i, CORPUS) for i in range(config.ENRON_SPLITS[s])] for s in ("certify", "utility")}
for split, units in REC_EN.items():
    assert [EN.manifest_row(u) for u in units] == json.load(open(config.SUITE_DATA / f"enron_manifest_{split}.json")), split
PARAMS_EN = json.load(open(config.REPORTS / "07_params.json"))
need = PL.prompt_rows(REC_EN)
need = need[need["retrieval"] == "pre"]["key"].unique()
rec_root_en = config.REPORTS / "11_records_enron"
for model in config.MODELS:
    cache = CL.ScoreCache(config.REPORTS / "07_scores" / f"{model}.jsonl", scorer=None)
    missing = [k for k in need if k not in cache.rows]
    assert not missing, f"{model}: {len(missing)} Domain 2 prompts were never scored; this notebook calls no model"
    AB.register(PARAMS_EN[model], cache, H.register_system)
    store = H.Store(rec_root_en, model)
    t_model = time.time()
    STATS[f"enron_{model}"] = {split: H.record_grid(units, SYSTEMS, None, model, store, workers=1, progress_every=10 ** 6) for split, units in REC_EN.items()}
    STATS[f"enron_{model}"]["compacted_bytes"] = store.compact()
    STATS[f"enron_{model}"]["seconds"] = round(time.time() - t_model, 1)
    print("enron", model, json.dumps(STATS[f"enron_{model}"]), flush=True)
ROWS = pd.concat([ROWS, pd.DataFrame([r for m in config.MODELS for r in H.Store(rec_root_en, m).all_rows() if r["system"] in SYSTEMS]).assign(domain="enron")],
                 ignore_index=True)
print("rows", ROWS.groupby(["domain", "model", "system"]).size().to_dict(), round(time.time() - t0), "s")

## Stage 3. Tables

In [ ]:
import sys, importlib
SRC = str(REPO_DIR / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()          # src/ may have been written after the bootstrap; the Drive mount does not refresh the importer cache
assert (REPO_DIR / "src" / "config.py").exists(), "src/config.py missing: run notebook 00 first"

# === STAGE 3: the full variant against notebook 04b, the ablation table, render modes, summary ===
import json
import config, ablation as AB, probe as PR, serving
for m in (config, AB, PR, serving): importlib.reload(m)
check = []
for domain, source in (("meridian", "04b_records"), ("enron", "07d_records")):
    for model in config.MODELS:
        ref = PR.load_rows_any(config.REPORTS / source, model, ["L2-RT-pre"])
        ref = ref[(ref["runtime"] == config.RUNTIME_VERSION) & ref["split"].isin(["certify", "utility"])]
        ref = {(r.uid, r.world, r.turn): r.text for r in ref.itertuples()}
        mine = ROWS[(ROWS.domain == domain) & (ROWS.model == model) & (ROWS.system == AB.system_name("full"))]
        keys = [(u, w, t) for u, w, t in zip(mine.uid, mine.world, mine.turn) if (u, w, t) in ref]
        same = sum(ref[k] == x for k, x in zip(zip(mine.uid, mine.world, mine.turn), mine.text) if k in ref)
        check.append({"domain": domain, "model": model, "source": source, "compared": len(keys), "identical": int(same),
                      "share": same / len(keys) if keys else float("nan")})
CHECK = pd.DataFrame(check)
display(CHECK)
assert (CHECK["share"] == 1.0).all() and (CHECK["compared"] > 0).all(), "the full variant does not reproduce the recorded selected configuration"
ABL = pd.concat([AB.ablation_table(ROWS[ROWS.domain == d]).assign(domain=d) for d in ("meridian", "enron")], ignore_index=True)
cols = ["domain", "model", "variant", "component", "resolved_all", "resolved_Q4", "resolved_Q5", "turns_mean", "wrong_answer_share", "asked_share",
        "identical_share", "named_restricted_answered_share", "named_restricted_asked_share", "attribute_questions", "routed_world1_S1", "routed_world0_S1"]
display(ABL[cols].round(3))
recs = {}
for model in config.MODELS:
    for mode in ("RT", "RS", "RF"):
        name = f"L2-{mode}-pre"
        df = PR.load_rows_any(config.REPORTS / "04b_records", model, [name])
        recs[(model, name)] = df[(df["runtime"] == config.RUNTIME_VERSION) & (df["split"] == "certify") & (df["kind"] == "leak") & (df["turn"] == 1)]
REND = AB.render_table(recs, pd.read_csv(config.REPORTS / "04b_certificate.csv"), pd.read_csv(config.REPORTS / "04b_utility.csv"))
display(REND.round(3))
ABL.to_csv(config.REPORTS / "11_ablation.csv", index=False)
REND.to_csv(config.REPORTS / "11_render.csv", index=False)
CHECK.to_csv(config.REPORTS / "11_full_vs_04b.csv", index=False)
summary = {"runtime_version": config.RUNTIME_VERSION, "dry_run": serving.DRY_RUN, "stats": STATS,
           "full_reproduces_04b": CHECK.to_dict("records"), "ablation": ABL[cols].round(4).to_dict("records"),
           "render": REND.round(4).to_dict("records")}
_ = (config.REPORTS / "11_summary.json").write_text(json.dumps(summary, indent=2, default=str))

## End of unit: commit

Save the notebook first (File > Save), then run the cell below.

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
subprocess.run(["git", "config", "core.logAllRefUpdates", "false"])   # Drive could not write .git/logs/HEAD; the reflog is not used
branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True).stdout.strip()
assert branch == "main", f"the repository is on branch {branch!r}, not main: switch to main before committing"
r = subprocess.run(["python", "tools/commit_cell.py",
    "nb11: ablations of L2-RT-pre replayed on the CPU from stored scores on Domain 1 and Domain 2 (score, conformal set, information-gain question, private action) and render modes from the 04b records; src/ablation.py, tests/test_ablation.py, tests/test_side.py (timing assertion no longer depends on the stand-in's speed); reports/11_records/, 11_records_enron/, 11_ablation.csv, 11_render.csv, 11_full_vs_04b.csv, 11_summary.json; docs/plan_v1.md amendments"],
    capture_output=True, text=True)
print(r.stdout); print(r.stderr)